In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "kanngiesser2020children")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Kanngiesser et al_2020_remove human data first_Respect ownership apes children_DevSci.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

In [3]:
df['study_id']="kanngiesser2020children"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)


In [4]:
df = df.rename(columns={"species": "species_original",
    "owner_id": "ape",
    "owner_gender": "owner_sex"})
df['role']='owner'
df = df.rename(columns={"nonowner_id": "ape_2"})
df['role_2']='non_owner'

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left') 

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
df= df.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')

df['dyad']=df.ape.str.cat(df.ape_2, sep='_')

df.columns
df.dropna(subset=['species'], inplace=True)

In [5]:
df.rename(columns={"ape": "participant", "ape_2":"participant_2"}, inplace=True)

In [6]:
complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
df= df.merge(subject_list,left_on='participant', right_on='name', how='left')
df.rename(columns={"age": "age_in_years"}, inplace=True)

complete_path_age = os.path.join(original_data_pathway, "subject_list_2.csv")
subject_list = pd.read_csv(complete_path_age)   
df= df.merge(subject_list,left_on='participant_2', right_on='name_2', how='left')
df.rename(columns={"age_2": "age_in_years_2"}, inplace=True)

df.rename(columns={"subgroup": "species_subgroup"}, inplace=True)

In [7]:
kanngiesser2020children_standardized=df[['study_id',
     'participant','age_in_years','sex','role',  'participant_2','age_in_years_2','sex_2','role_2','species','genus','dyad','dyad_id', 'session ', 'trial', 'trial_id',
      'species_subgroup', 'condition','owner_dom',
       'start_individ', 'owner_sex',  'own_items', 'other_items', 'total_owner']]
comp_out_path_stand = os.path.join(out_pathway, 'kanngiesser2020children_exp1_standardized.csv')
kanngiesser2020children_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

In [8]:

names =kanngiesser2020children_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
kanngiesser2020children_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'kanngiesser2020children_exp1_glossary.csv')
kanngiesser2020children_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)